# 03 · Features: pre-game inputs for the points model

**Goal (Phase 3):** for every player-game row, build features that use **only games before this one**.

Pattern: `groupby('player_id')` → `shift(1)` → `rolling(...)`. `shift(1)` keeps tonight's own result out (no leakage).

| Group | Features |
|---|---|
| Player form | rolling points / minutes / FGA / 3PA / FTA over last 5, last 10, season-to-date |
| Role | minutes trend; points per minute; usage |
| Teammates | number of starters out; points per game of missing teammates |
| Schedule | home/away, rest days, back-to-back |
| Opponent | opponent defensive rating and pace (rolling, pre-game) |
| Market | game spread and total (later) |

Input: `nba/data/processed/games.parquet` (sorted by date, player_id). Output: `nba/data/processed/features.parquet`.

In [16]:
import pandas as pd
import numpy as np
from fbml.config import ROOT

pd.set_option("display.max_columns", 50)

PROCESSED = ROOT / "nba" / "data" / "processed"
games = pd.read_parquet(PROCESSED / "games.parquet")

# rolling features assume each player's rows are in time order
games = games.sort_values(["player_id", "date"]).reset_index(drop=True)
print(games.shape)
games.head()

(130830, 13)


,season,player_id,player,team,opponent,game_id,date,home,minutes,points,fga,fg3a,fta
0,22021,2207,Joe Johnson,BOS,CLE,0022100471,2021-12-22,1,2,2,1,0,0
1,22021,2544,LeBron James,LAL,GSW,0022100002,2021-10-19,1,37,34,23,11,6
2,22021,2544,LeBron James,LAL,PHX,0022100025,2021-10-22,1,37,25,18,9,4
3,22021,2544,LeBron James,LAL,MEM,0022100040,2021-10-24,1,40,19,19,9,2
4,22021,2544,LeBron James,LAL,CLE,0022100077,2021-10-29,1,38,26,22,10,5


In [17]:

games['pts_last5'] = games.groupby('player_id')['points'].transform(lambda s : s.shift(1).rolling(5).mean())
games[games['player'] == 'LeBron James'][['date', 'player', 'points', 'pts_last5']].head(7)

,date,player,points,pts_last5
1,2021-10-19,LeBron James,34,NaN
2,2021-10-22,LeBron James,25,NaN
3,2021-10-24,LeBron James,19,NaN
4,2021-10-29,LeBron James,26,NaN
5,2021-10-31,LeBron James,15,NaN
6,2021-11-02,LeBron James,30,23.8
7,2021-11-19,LeBron James,23,23.0


In [18]:
# rolling form: mean of each stat over the player's previous 5 and 10 games
# shift(1) keeps tonight out; NaN until enough earlier games exist
FORM_STATS = ['points', 'minutes', 'fga', 'fg3a', 'fta']
WINDOWS = [5, 10]

for stat in FORM_STATS:
    for window in WINDOWS:
        games[f"{stat}_last{window}"] = games.groupby('player_id')[stat].transform(
            lambda s, w=window: s.shift(1).rolling(w).mean()
        )

games = games.drop(columns='pts_last5')  # replaced by points_last5
games[games['player'] == 'LeBron James'][['date', 'points', 'points_last5', 'points_last10']].head(12)

,date,points,points_last5,points_last10
1,2021-10-19,34,NaN,NaN
2,2021-10-22,25,NaN,NaN
3,2021-10-24,19,NaN,NaN
4,2021-10-29,26,NaN,NaN
5,2021-10-31,15,NaN,NaN
6,2021-11-02,30,23.8,NaN
7,2021-11-19,23,23.0,NaN
8,2021-11-21,10,22.6,NaN
9,2021-11-24,39,20.8,NaN
10,2021-11-26,30,23.4,NaN


In [ ]:
# season-to-date: mean of all earlier games THIS season (resets each season)
games['points_season_avg'] = games.groupby(['player_id', 'season'])['points'].transform(
    lambda s: s.shift(1).expanding().mean()
)

# check around LeBron's season change (56 games in 2021-22)
lebron = games[games['player'] == 'LeBron James']
lebron[['season', 'date', 'points', 'points_last5', 'points_season_avg']].iloc[54:60]

In [20]:
# scoring rate: total points / total minutes over the previous 10 games
games['pts_per_min_last10'] = games['points_last10'] / games['minutes_last10']
print(games['pts_per_min_last10'].describe().round(3))

# sanity check: top rates among real rotation players (25+ min per game lately)
rotation = games[games['minutes_last10'] >= 25]
rotation.sort_values('pts_per_min_last10', ascending=False)[['player', 'date', 'minutes_last10', 'pts_per_min_last10']].head(5)

count    121468.000
mean          0.443
std           0.156
min           0.000
25%           0.332
50%           0.425
75%           0.537
max           1.182
Name: pts_per_min_last10, dtype: float64


,player,date,minutes_last10,pts_per_min_last10
24708,Joel Embiid,2023-12-22,33.5,1.182090
24710,Joel Embiid,2024-01-05,33.3,1.180180
24709,Joel Embiid,2024-01-02,33.3,1.177177
19783,Giannis Antetokounmpo,2023-02-14,31.9,1.169279
52893,Shai Gilgeous-Alexander,2025-02-07,33.0,1.163636


In [21]:
# days since the player's previous game (NaN for his first game in the data)
games['rest_days'] = games.groupby('player_id')['date'].diff().dt.days

# first game of each season: its 'rest' is the summer break, not a missed stretch
first_of_season = games.groupby(['player_id', 'season']).cumcount() == 0

games['back_to_back'] = (games['rest_days'] == 1).astype(int)
games['long_rest'] = ((games['rest_days'] >= 4) & ~first_of_season).astype(int)

print("back_to_back share:", round(games['back_to_back'].mean(), 3))
print("long_rest share:   ", round(games['long_rest'].mean(), 3))
games.groupby('long_rest')[['points', 'minutes']].mean().round(2)

back_to_back share: 0.146
long_rest share:    0.149


,points,minutes
long_rest,,
0,11.23,23.78
1,7.68,16.57


In [ ]:
# Part 1: the team's usual scoring = its average total points over the previous 10 games
team_games = (
    games.groupby(['team', 'game_id', 'date'], as_index=False)['points'].sum()
    .rename(columns={'points': 'team_pts'})
    .sort_values(['team', 'date'])
)
team_games['team_pts_last10'] = team_games.groupby('team')['team_pts'].transform(
    lambda s: s.shift(1).rolling(10).mean()
)
print(team_games['team_pts_last10'].describe().round(1))

In [ ]:
# Part 2: how much usual scoring is missing tonight
games = games.merge(team_games[['team', 'game_id', 'team_pts_last10']], on=['team', 'game_id'], how='left')

# usual scoring of the players who actually played tonight (sum of their points_last10)
available_pts = games.groupby(['team', 'game_id'])['points_last10'].transform('sum')
games['missing_pts'] = games['team_pts_last10'] - available_pts

print(games['missing_pts'].describe().round(1))

# does it work? when more scoring is missing, players beat their own recent average
gap = games['points'] - games['points_last10']
buckets = pd.cut(games['missing_pts'], [-100, 0, 10, 20, 30, 200])
games.groupby(buckets, observed=True).agg(rows=('points', 'size'), pts_above_usual=('points', lambda x: gap[x.index].mean())).round(2)

In [ ]:
# team box scores (one row per team per game) for usage, pace and defence
team_files = sorted((ROOT / "nba" / "data" / "raw").glob("team_gamelogs_*.csv"))
teams = pd.concat([pd.read_csv(f, dtype={"GAME_ID": str}) for f in team_files], ignore_index=True)
teams = teams.rename(columns={'TEAM_ABBREVIATION': 'team', 'GAME_ID': 'game_id', 'GAME_DATE': 'date',
                              'MIN': 'team_min', 'FGA': 't_fga', 'FTA': 't_fta', 'OREB': 't_oreb',
                              'TOV': 't_tov', 'PTS': 't_pts'})
teams['date'] = pd.to_datetime(teams['date'])

# possessions estimate (standard box-score formula)
teams['poss'] = teams['t_fga'] + 0.44 * teams['t_fta'] - teams['t_oreb'] + teams['t_tov']

# attach the opponent's row from the same game: their points = points this team allowed
opp = teams[['game_id', 'team', 't_pts', 'poss']].rename(
    columns={'team': 'opp_team', 't_pts': 'pts_allowed', 'poss': 'opp_poss'})
teams = teams.merge(opp, on='game_id')
teams = teams[teams['team'] != teams['opp_team']]

game_poss = (teams['poss'] + teams['opp_poss']) / 2
teams['def_rating'] = 100 * teams['pts_allowed'] / game_poss      # points allowed per 100 possessions
teams['pace'] = game_poss * 240 / teams['team_min']               # possessions per 48 minutes (OT-adjusted)
print(teams.shape)
teams[['def_rating', 'pace']].describe().round(1)

In [ ]:
# Role: minutes trend and usage share
# trend > 0: playing more lately than over the last 10 (role growing)
games['minutes_trend'] = games['minutes_last5'] - games['minutes_last10']

# usage: share of the team's shot attempts this player takes (FTA weighted 0.44, like possessions)
games = games.merge(teams[['team', 'game_id', 't_fga', 't_fta']], on=['team', 'game_id'], how='left')
usage_tonight = (games['fga'] + 0.44 * games['fta']) / (games['t_fga'] + 0.44 * games['t_fta'])
games['usage_last10'] = usage_tonight.groupby(games['player_id']).transform(
    lambda s: s.shift(1).rolling(10).mean()
)
games = games.drop(columns=['t_fga', 't_fta'])

print(games[['minutes_trend', 'usage_last10']].describe().round(3))
rotation = games[games['minutes_last10'] >= 25]
rotation.sort_values('usage_last10', ascending=False)[['player', 'date', 'usage_last10']].head(5)

In [ ]:
# Opponent: rolling defence and pace over the opponent's previous 10 games
teams = teams.sort_values(['team', 'date'])
for col in ['def_rating', 'pace']:
    teams[f'{col}_last10'] = teams.groupby('team')[col].transform(lambda s: s.shift(1).rolling(10).mean())

opp_form = teams[['team', 'game_id', 'def_rating_last10', 'pace_last10']].rename(
    columns={'team': 'opponent', 'def_rating_last10': 'opp_def_rating_last10', 'pace_last10': 'opp_pace_last10'})
games = games.merge(opp_form, on=['opponent', 'game_id'], how='left')

# sanity check: do players score more against weak defences (high rating = more points allowed)?
gap = games['points'] - games['points_last10']
q = pd.qcut(games['opp_def_rating_last10'], 5, labels=['best D', '2', '3', '4', 'worst D'])
gap.groupby(q, observed=True).mean().round(2)

In [ ]:
# Final feature table
FEATURES = (
    [f'{s}_last{w}' for s in FORM_STATS for w in WINDOWS]
    + ['points_season_avg', 'pts_per_min_last10', 'minutes_trend', 'usage_last10',
       'home', 'rest_days', 'back_to_back', 'long_rest',
       'team_pts_last10', 'missing_pts',
       'opp_def_rating_last10', 'opp_pace_last10']
)
ID_COLS = ['season', 'player_id', 'player', 'team', 'opponent', 'game_id', 'date']
TARGET = 'points'

features = games[ID_COLS + FEATURES + [TARGET, 'minutes']].sort_values(['date', 'player_id']).reset_index(drop=True)

# leakage guard: every feature must be known before tip-off, so none may equal tonight's points exactly
assert not any((features[f] == features[TARGET]).all() for f in FEATURES)

print(features.shape, "|", len(FEATURES), "features")
print(features[FEATURES].isna().mean().round(3).sort_values(ascending=False).head(8))

features.to_parquet(PROCESSED / "features.parquet", index=False)
print("saved ->", PROCESSED / "features.parquet")

## Phase 3 summary
- **Form:** points / minutes / FGA / 3PA / FTA over the last 5 and 10 games; season-to-date points (resets each season).
- **Role:** points per minute (last 10), minutes trend (last 5 − last 10), usage share of team shots (last 10).
- **Schedule:** home, rest days, back-to-back, long rest (4+ days, not counting season openers).
- **Teammates:** team's usual scoring and `missing_pts` (usual scoring of players not playing tonight).
- **Opponent:** defensive rating and pace over its previous 10 games.
- **Not yet:** market spread/total (needs odds data, later).
- NaNs are left in (early-career / early-season rows). Phase 4 decides how to handle them (the roadmap says skip each player's first ~5 games or fall back to last season).